<a href="https://colab.research.google.com/github/Codegea/AplicacionesHibridas/blob/master/TransfAnalisBasicPython.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Lectura de una archivo .xlsl

In [46]:
import pandas as pd # 1. importamos la libreria de pandas.

In [47]:
from google.colab import files # 2. Para subir el archivo desd la PC.
uploaded = files.upload()

df = pd.read_excel('excel_actividad.xlsx', sheet_name='Datos_Crudos') # 3. Le decimos exactamente cual hoja del .xlsx queremos que lea

Saving excel_actividad.xlsx to excel_actividad (2).xlsx


In [48]:
df.head() # 4. Lee todo el archivo y muestra los primero 5 registros

,id_estudiante,nombre_estudiante,programa,asignatura,grupo,tipo_actividad,fecha_entrega_raw,horas_estudio_raw,modalidad_entrega_raw,nota_quiz,nota_taller,meta_horas,asistencia_pct
0,EST001,Ana Torres,Ingeniería de Datos,Programación,C,Proyecto,13/04/26,5h,presencial,4.2,4.1,6,88
1,EST002,Luis Pérez,Psicología,Programación,A,Taller,13/04/26,5h,remota,3.8,3.8,12,77
2,EST003,María Gómez,Psicología,Visualización de Datos,B,Taller,12-04-2026,ocho,Híbrida,2.8,2.9,10,73
3,EST004,Carlos Rojas,Ingeniería de Datos,Metodología,A,Proyecto,15-04-26,dos horas,hibrida,4.4,4.3,6,99
4,EST005,Laura Martínez,Psicología,Programación,C,Proyecto,19/04/26,7,Presencial,4.1,2.4,8,94


In [49]:
df.shape # 5. Nos dice cuantas filas y columnas tiene nuestro archivo


(36, 13)

In [50]:
list(df.columns) # 6.Lista el nombre de las columnas
df.dtypes # 7. Muestra los tipos de datos

,0
id_estudiante,object
nombre_estudiante,object
programa,object
asignatura,object
grupo,object
tipo_actividad,object
fecha_entrega_raw,object
horas_estudio_raw,object
modalidad_entrega_raw,object
nota_quiz,float64


In [51]:
df['fecha_entrega_raw'].head(10) # Esto es para identificar el formato en el que viene cada una de las fechas.

,fecha_entrega_raw
0,13/04/26
1,13/04/26
2,12-04-2026
3,15-04-26
4,19/04/26
5,16/04/2026
6,19/04/26
7,10/04/2026
8,16/04/2026
9,15-04-26


# Transformación 1. Fecha de entrega

In [52]:
df['fecha_entrega_raw'] = df['fecha_entrega_raw'].str.replace('-', '/', regex=False) # Unifica las fechas cambia todos los separadores entre fechas - y les pone /

# Crea una nueva columna, le agrega un tipo de dato fecha y le da un formato unificado
df['fecha_entrega'] = pd.to_datetime(
    df['fecha_entrega_raw'],
    dayfirst=True,
    errors='coerce'
)

/tmp/ipykernel_4921/562303058.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['fecha_entrega'] = pd.to_datetime(


In [53]:
df[df['fecha_entrega'].isna()] # Para saber si algún registro falló

,id_estudiante,nombre_estudiante,programa,asignatura,grupo,tipo_actividad,fecha_entrega_raw,horas_estudio_raw,modalidad_entrega_raw,nota_quiz,nota_taller,meta_horas,asistencia_pct,fecha_entrega


In [54]:
df['fecha_entrega_fmt'] = df['fecha_entrega'].dt.strftime('%d/%m/%Y') # Le da formato dd/mm/aaaa
df[['fecha_entrega_raw', 'fecha_entrega_fmt']].head(10) # Muestra los 10 primero registros

,fecha_entrega_raw,fecha_entrega_fmt
0,13/04/26,13/04/2026
1,13/04/26,13/04/2026
2,12/04/2026,12/04/2026
3,15/04/26,15/04/2026
4,19/04/26,19/04/2026
5,16/04/2026,16/04/2026
6,19/04/26,19/04/2026
7,10/04/2026,10/04/2026
8,16/04/2026,16/04/2026
9,15/04/26,15/04/2026


# Transformación 2. Horas de estudio

In [55]:
import re

mapa_numeros = {
    'cero': 0, 'uno': 1, 'dos': 2, 'tres': 3, 'cuatro': 4,
    'cinco': 5, 'seis': 6, 'siete': 7, 'ocho': 8, 'nueve': 9,
    'diez': 10
}

def convertir_horas(valor):
    if pd.isna(valor):
        return None

    texto = str(valor).lower().strip()

    # 1. Buscar número directo (ej: 5, 7, etc.)
    numero = re.search(r'\d+', texto)
    if numero:
        return int(numero.group())

    # 2. Buscar número en texto (ej: "dos horas")
    for palabra, num in mapa_numeros.items():
        if palabra in texto:
            return num

    return None

df['horas_estudio'] = df['horas_estudio_raw'].apply(convertir_horas)

In [56]:
df[['horas_estudio_raw', 'horas_estudio']].head(10)

,horas_estudio_raw,horas_estudio
0,5h,5
1,5h,5
2,ocho,8
3,dos horas,2
4,7,7
5,6 horas,6
6,4,4
7,5h,5
8,9 h,9
9,5h,5


In [57]:
df[df['horas_estudio'].isna()]

,id_estudiante,nombre_estudiante,programa,asignatura,grupo,tipo_actividad,fecha_entrega_raw,horas_estudio_raw,modalidad_entrega_raw,nota_quiz,nota_taller,meta_horas,asistencia_pct,fecha_entrega,fecha_entrega_fmt,horas_estudio


In [58]:
df[['horas_estudio_raw', 'horas_estudio']]

,horas_estudio_raw,horas_estudio
0,5h,5
1,5h,5
2,ocho,8
3,dos horas,2
4,7,7
5,6 horas,6
6,4,4
7,5h,5
8,9 h,9
9,5h,5


# Transformación 3. Modalidad de entrega

In [59]:
def clasificar_modalidad(valor):
    if pd.isna(valor):
        return None

    texto = str(valor).lower().strip()

    # limpiar espacios múltiples
    texto = ' '.join(texto.split())

    # clasificación
    if 'presencial' in texto:
        return 'Presencial'
    elif 'virtual' in texto:
        return 'Virtual'
    elif 'hibrid' in texto or 'híbrida' in texto:
        return 'Híbrida'
    elif 'mixta' in texto:
        return 'Mixta'
    elif 'remota' in texto:
        return 'Remota'
    elif 'linea' in texto or 'línea' in texto:
        return 'En linea'

    return 'Otro'

df['modalidad_entrega'] = df['modalidad_entrega_raw'].apply(clasificar_modalidad)

In [60]:
df[['modalidad_entrega_raw', 'modalidad_entrega']]

,modalidad_entrega_raw,modalidad_entrega
0,presencial,Presencial
1,remota,Remota
2,Híbrida,Híbrida
3,hibrida,Híbrida
4,Presencial,Presencial
5,en linea,En linea
6,remota,Remota
7,virtual,Virtual
8,presencial,Presencial
9,presencial,Presencial


# Parte 3. Cálculos solicitados